# SyTF — Symbolic Time-series Forecasting (simulated random walk)

Fits PySR on a simulated random-walk series (200 observations, lag 1, 180 train / 20 test).


In [ ]:
import os
import sys
from pathlib import Path

# PySR/Julia: disable stdio interception when supported
os.environ.setdefault("PYTHONCALL_STDIO", "0")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from darts import TimeSeries
from darts.metrics import metrics

from src.config import DATA_DIR, ensure_results_dir
from src.data_utils import series_to_supervised
from src.syntf.pysr_runner import fit_syntf


In [ ]:
# Load simulated random-walk data
lags = 1
run_name = f"SyTF_RW_Simulated_lag{lags}"
train_size = 180
test_size = 20

data_path = DATA_DIR / "RW_Simulated_datset.csv"
if not data_path.exists():
    raise FileNotFoundError(
        f"Missing {data_path.name}. Run: python scripts/generate_rw_simulated.py"
    )

df = pd.read_csv(data_path)
print(df.head())
print("Number of rows:", len(df))

dat = df["x"].values.reshape(-1, 1)
tr_dat = dat[:train_size]
ts_dat = dat[train_size:]
fore_hor = len(ts_dat)


In [ ]:
# Build supervised training matrix
tr_data_supervised = series_to_supervised(tr_dat, n_in=lags, n_out=1)
train_data = tr_data_supervised.reset_index(drop=True)

X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]

# Clean column names for PySR
X_train.columns = X_train.columns.str.replace(r"[^0-9a-zA-Z_]+", "", regex=True)

print("X_train columns:", X_train.columns.tolist())
print(X_train.head())


In [ ]:
# Fit SyTF (PySR)
model, final_eq, train_time = fit_syntf(
    X_train,
    y_train,
    n_lags=lags,
    niterations=25,
    run_name=run_name,
)

print("Sympy equation:")
print(final_eq)
print(f"Training time: {train_time:.2f} s")


In [ ]:
# Training predictions
train_pred = model.predict(X_train.values)

plt.figure()
plt.plot(train_pred, label="train_pred")
plt.plot(y_train.values.flatten(), label="y_train")
plt.legend()
plt.title("SyTF training fit")
plt.show()


In [ ]:
# Test set (last 20 points)
full_data = series_to_supervised(dat, n_in=lags, n_out=1)
test_data = full_data.tail(len(ts_dat)).reset_index(drop=True)

X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
X_test.columns = X_test.columns.str.replace(r"[^0-9a-zA-Z_]+", "", regex=True)

start_time = time.time()
test_pred_one = model.predict(X_test.values)
pred_time = time.time() - start_time
total_time = train_time + pred_time

model_name = "SyTF_lag1"
print(f"Total time (train + test prediction): {total_time:.2f} s")

plt.figure()
plt.plot(test_pred_one, label="test_pred")
plt.plot(y_test.values.flatten(), label="y_test")
plt.legend()
plt.title("SyTF test fit")
plt.show()


In [ ]:
# Metrics
smape = metrics.smape(
    TimeSeries.from_values(np.array(y_test)),
    TimeSeries.from_values(test_pred_one),
)
mae = metrics.mae(
    TimeSeries.from_values(np.array(y_test)),
    TimeSeries.from_values(test_pred_one),
)
rmse = metrics.rmse(
    TimeSeries.from_values(np.array(y_test)),
    TimeSeries.from_values(test_pred_one),
)
marre = metrics.marre(
    TimeSeries.from_values(np.array(y_test)),
    TimeSeries.from_values(test_pred_one),
)

print(f"SMAPE: {smape:.4f}")
print(f"MAE:   {mae:.4f}")
print(f"RMSE:  {rmse:.4f}")
print(f"MARRE: {marre:.4f}")


In [ ]:
# Save test predictions
out_dir = ensure_results_dir(run_name)
pred_path = out_dir / f"{run_name}_predictions.csv"
pd.DataFrame({"y_test": y_test.values, "y_pred": test_pred_one}).to_csv(pred_path, index=False)
print(f"Saved predictions to {pred_path}")
